<center>Universidad de los Andes - Maestría en Ingeniería de Sistemas y Computación<br>
Algoritmos de Computación Cuantica <br>
Jose Emanuel Toro Medina</center>


# Taller: Centro de Colombia y EE. UU. (Optimizado)
Para poder ejecutar el siguiente taller/codigos por favor seguir las siguientes intrucciones/"Tabla de contenidos".
 1. Importación y limpieza de datos (Se leen las bases de datos de excel en las siguientes rutas; bases/DIVIPOLA_Municipios.xlsx y bases/uscounties.xlsx).
 2. Algoritmo de Welzl (Optimizado - O(n) esperado)
 3. Ejecución Completa con Welzl (Colombia y USA)
 4. Algoritmo Naïve del Círculo Mínimo Contenedor 
 5. Comparación de Rendimiento: Welzl vs Naïve (Se recomienda ejecutar para lograr ver y entender la diferencia de complejidad)

## 1. Importación y Limpieza de Datos

Importante respetar/cambiar las rutas en la que se leen las bases:

**Lecutra actual:**

- bases/DIVIPOLA_Municipios.xlsx
- bases/uscounties.xlsx

In [43]:
import pandas as pd
import numpy as np
import time

def cargar_colombia():
    path_col = 'bases/DIVIPOLA_Municipios.xlsx'
    df_col = pd.read_excel(path_col, skiprows=9)
    df_col.columns = ['dpt_code', 'dpt_name', 'mun_code', 'mun_name', 'type', 'lat', 'lon', 'extra']
    
    df_col = df_col.dropna(subset=['lat', 'lon'])
    df_col = df_col[df_col['lat'] != 'LATITUD']
    
    df_col['lat'] = df_col['lat'].astype(float)
    df_col['lon'] = df_col['lon'].astype(float)
    
    z_colombia = df_col['lon'].values + 1j * df_col['lat'].values
    print(f"Colombia: Cargados {len(z_colombia)} municipios correctamente.")
    return z_colombia, df_col

def cargar_usa():
    path_usa = 'bases/uscounties.xlsx'
    df_usa = pd.read_excel(path_usa)
    
    df_usa = df_usa.dropna(subset=['lat', 'lng'])
    df_usa['lat'] = df_usa['lat'].astype(float)
    df_usa['lng'] = df_usa['lng'].astype(float)
    
    z_usa = df_usa['lng'].values + 1j * df_usa['lat'].values
    print(f"USA: Cargados {len(z_usa)} condados correctamente.")
    return z_usa, df_usa

colombia_points, df_colombia = cargar_colombia()
usa_points, df_usa = cargar_usa()

Colombia: Cargados 1122 municipios correctamente.
USA: Cargados 3144 condados correctamente.


## 2. Algoritmo de Welzl (Optimizado - O(n) esperado)

El **algoritmo de Welzl** es un algoritmo recursivo y aleatorio que encuentra el círculo mínimo contenedor en tiempo esperado **O(n)**, mucho más rápido que el naïve O(n⁴).

### Idea General:
1. **Mezcla aleatoria**: Los puntos se procesan en orden aleatorio.
2. **Recursión**: Para cada punto:
   - Si está dentro del MEC actual, continúa.
   - Si está fuera, entonces **debe estar en la frontera** del nuevo MEC.

### Ventajas:
- **Tiempo esperado O(n)** vs O(n⁴) del naïve.
- Funciona bien en práctica incluso para grandes conjuntos.
- Solo mantiene hasta 3 puntos en la frontera.


In [44]:
def obtener_circuncentro(z1, z2, z3):
    A = z2 - z1
    B = z3 - z1
    num = B * (A * A.conjugate()) - A * (B * B.conjugate())
    den = B * A.conjugate() - A * B.conjugate()
    
    if abs(den) < 1e-12:
        return None, None
    
    W = num / den
    c = z1 + W
    r = abs(c - z1)
    return c, r


import random
import sys

sys.setrecursionlimit(10000)

# --- 4. IMPLEMENTACIÓN DEL ALGORITMO DE WELZL (OPTIMIZADO) ---

def obtener_circulo_de_frontera(frontera):
    if len(frontera) == 0:
        return 0 + 0j, -1
    elif len(frontera) == 1:
        return frontera[0], 0
    elif len(frontera) == 2:
        c = (frontera[0] + frontera[1]) / 2.0
        r = abs(frontera[0] - frontera[1]) / 2.0
        return c, r
    else: 
        c, r = obtener_circuncentro(frontera[0], frontera[1], frontera[2])
        if r is None: 
            opciones = [
                (frontera[0], frontera[1]),
                (frontera[0], frontera[2]),
                (frontera[1], frontera[2]),
            ]
            a, b = max(opciones, key=lambda par: abs(par[0] - par[1]))
            c = (a + b) / 2.0
            r = abs(a - b) / 2.0
        return c, r

def dentro_del_circulo(punto, centro, radio, tolerancia=1e-9):
    if radio is None or radio < 0:
        return False
    return abs(punto - centro) <= radio + tolerancia

def _circulo_diametro(a, b):
    return (a + b) / 2.0, abs(a - b) / 2.0

def _circulo_circunscrito(a, b, c):
    centro, radio = obtener_circuncentro(a, b, c)
    if radio is None:
        return None
    return centro, radio

def _producto_cruz(a, b, c):
    return (b.real - a.real) * (c.imag - a.imag) - (b.imag - a.imag) * (c.real - a.real)

def _circulo_un_punto(puntos, p):
    c, r = p, 0.0
    for i, q in enumerate(puntos):
        if abs(q - c) > r + 1e-9:
            if r == 0.0:
                c, r = _circulo_diametro(p, q)
            else:
                c, r = _circulo_dos_puntos(puntos[:i + 1], p, q)
    return c, r

def _circulo_dos_puntos(puntos, p, q):
    circ = _circulo_diametro(p, q)
    izquierda = None
    derecha = None
    for r_punto in puntos:
        if abs(r_punto - circ[0]) <= circ[1] + 1e-9:
            continue
        cruz = _producto_cruz(p, q, r_punto)
        cand = _circulo_circunscrito(p, q, r_punto)
        if cand is None:
            continue
        if cruz > 0.0 and (izquierda is None or
                           _producto_cruz(p, q, cand[0]) > _producto_cruz(p, q, izquierda[0])):
            izquierda = cand
        elif cruz < 0.0 and (derecha is None or
                             _producto_cruz(p, q, cand[0]) < _producto_cruz(p, q, derecha[0])):
            derecha = cand
    if izquierda is None and derecha is None:
        return circ
    if izquierda is None:
        return derecha
    if derecha is None:
        return izquierda
    return izquierda if izquierda[1] <= derecha[1] else derecha

def welzl_recursivo(puntos, frontera):
    if len(puntos) == 0 or len(frontera) == 3:
        c, r = obtener_circulo_de_frontera(frontera)
        return c, r, list(frontera)
    centro_final, radio_final, frontera_final = encontrar_mec_welzl(puntos)
    return centro_final, radio_final, frontera_final

def encontrar_mec_welzl(puntos):
    puntos_lista = list(puntos)
    random.shuffle(puntos_lista)

    if not puntos_lista:
        return 0 + 0j, -1, []

    centro_total = None
    radio_total = None
    for i, p in enumerate(puntos_lista):
        if centro_total is None or abs(p - centro_total) > radio_total + 1e-9:
            centro_total, radio_total = _circulo_un_punto(puntos_lista[:i + 1], p)

    radio_total = float(radio_total)

    frontera = [p for p in puntos_lista if abs(abs(p - centro_total) - radio_total) < 1e-6]

    return centro_total, radio_total, frontera


## 3 Ejecución Completa con Welzl (Colombia y USA)

Ejecución del algoritmo de Welzl con **todos los datos** de Colombia y USA.


In [45]:
# Colombia Completa
print("=" * 70)
print("EJECUCIÓN COMPLETA: COLOMBIA")
print("=" * 70)
print(f"Total de municipios: {len(colombia_points)}")

t0 = time.time()
centro_col_welzl, radio_col_welzl, frontera_col_welzl = encontrar_mec_welzl(colombia_points)
t1 = time.time()

print(f"Tiempo de ejecución: {t1 - t0:.4f} segundos")
print(f"Centro (complejo): {centro_col_welzl}")
print(f"Centro (lat, lon): ({centro_col_welzl.imag:.6f}, {centro_col_welzl.real:.6f})")
print(f"Radio: {radio_col_welzl:.6f} grados")
print(f"Puntos en la frontera: {len(frontera_col_welzl)}")
print()

# USA Completa
print("=" * 70)
print("EJECUCIÓN COMPLETA: USA")
print("=" * 70)
print(f"Total de condados: {len(usa_points)}")

t0 = time.time()
centro_usa_welzl, radio_usa_welzl, frontera_usa_welzl = encontrar_mec_welzl(usa_points)
t1 = time.time()

print(f"Tiempo de ejecución: {t1 - t0:.4f} segundos")
print(f"Centro (complejo): {centro_usa_welzl}")
print(f"Centro (lat, lon): ({centro_usa_welzl.imag:.6f}, {centro_usa_welzl.real:.6f})")
print(f"Radio: {radio_usa_welzl:.6f} grados")
print(f"Puntos en la frontera: {len(frontera_usa_welzl)}")


EJECUCIÓN COMPLETA: COLOMBIA
Total de municipios: 1122
Tiempo de ejecución: 0.0216 segundos
Centro (complejo): (-75.7095111995+4.910525408j)
Centro (lat, lon): (4.910525, -75.709511)
Radio: 10.165068 grados
Puntos en la frontera: 2

EJECUCIÓN COMPLETA: USA
Total de condados: 3144
Tiempo de ejecución: 0.0132 segundos
Centro (complejo): (-117.24881162141486+48.08172406217765j)
Centro (lat, lon): (48.081724, -117.248812)
Radio: 49.713823 grados
Puntos en la frontera: 3


## 4. Algoritmo Naïve del Círculo Mínimo Contenedor (MEC)

In [46]:
def contiene_todos(c, r, puntos):
    """
    Verifica si todos los puntos están dentro del círculo definido por (c, r).
    Usa una tolerancia para evitar errores de precisión numérica.
    """
    tolerancia = 1e-9
    for p in puntos:
        if abs(p - c) > r + tolerancia:
            return False
    return True

def encontrar_mec_naive(puntos):
    """
    Busca el círculo mínimo contenedor (MEC) recorriendo exhaustivamente
    todas las parejas y ternas posibles.
    """
    n = len(puntos)
    R_min = float('inf')
    c_best = None
    puntos_frontera = []

    for i in range(n):
        for j in range(i + 1, n):
            z1, z2 = puntos[i], puntos[j]
            c = (z1 + z2) / 2.0
            r = abs(z1 - z2) / 2.0
            
            if contiene_todos(c, r, puntos):
                if r < R_min:
                    R_min = r
                    c_best = c
                    puntos_frontera = [z1, z2]

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                z1, z2, z3 = puntos[i], puntos[j], puntos[k]
                c, r = obtener_circuncentro(z1, z2, z3)
                
                if r is not None:
                    if contiene_todos(c, r, puntos):
                        if r < R_min:
                            R_min = r
                            c_best = c
                            puntos_frontera = [z1, z2, z3]
                            
    return c_best, R_min, puntos_frontera


In [47]:
import time

# Tomamos una muestra de los primeros n municipios para una prueba rápida
puntos_prueba = colombia_points[:100]

t0 = time.time()
centro, radio, frontera = encontrar_mec_naive(puntos_prueba)
t1 = time.time()

print(f"Prueba completada en: {t1 - t0:.4f} segundos.")
print(f"Centro encontrado: {centro}")
print(f"Radio: {radio}")
print(f"Puntos en la frontera (complejos): {frontera}")

# Descomentar para hacer la prueba total del algoritmo

'''t0 = time.time()
centro_col, radio_col, frontera_col = encontrar_mec_naive(colombia_points)
t1 = time.time()
print(f"Colombia completa resuelta en: {t1 - t0:.4f} segundos.")'''

Prueba completada en: 0.6633 segundos.
Centro encontrado: (-75.863526264+7.1442084475000005j)
Radio: 1.7000771222726672
Puntos en la frontera (complejos): [np.complex128(-75.194570185+5.581275312j), np.complex128(-76.532482343+8.707141583j)]


't0 = time.time()\ncentro_col, radio_col, frontera_col = encontrar_mec_naive(colombia_points)\nt1 = time.time()\nprint(f"Colombia completa resuelta en: {t1 - t0:.4f} segundos.")'

## 5. Comparación de Rendimiento: Welzl vs Naïve

A continuación se comparan ambos algoritmos en términos de tiempo de ejecución y resultados.


In [48]:
# Prueba con subconjunto pequeño
puntos_test = colombia_points[:100]

print("=" * 60)
print("PRUEBA CON 100 MUNICIPIOS DE COLOMBIA")
print("=" * 60)

# Algoritmo Naïve
print("\n1. ALGORITMO NAÏVE (O(n^4)):")
t0 = time.time()
centro_naive, radio_naive, frontera_naive = encontrar_mec_naive(puntos_test)
t1 = time.time()
tiempo_naive = t1 - t0
print(f"   Tiempo: {tiempo_naive:.4f} segundos")
print(f"   Centro: {centro_naive}")
print(f"   Radio: {radio_naive:.6f}")
print(f"   Puntos frontera: {len(frontera_naive)}")

# Algoritmo Welzl
print("\n2. ALGORITMO DE WELZL (O(n) esperado):")
t0 = time.time()
centro_welzl, radio_welzl, frontera_welzl = encontrar_mec_welzl(puntos_test)
t1 = time.time()
tiempo_welzl = t1 - t0
print(f"   Tiempo: {tiempo_welzl:.4f} segundos")
print(f"   Centro: {centro_welzl}")
print(f"   Radio: {radio_welzl:.6f}")
print(f"   Puntos frontera: {len(frontera_welzl)}")

# Comparación
print("\n" + "=" * 60)
print(f"ACELERACIÓN: {tiempo_naive / tiempo_welzl:.2f}x más rápido con Welzl")
print("=" * 60)


PRUEBA CON 100 MUNICIPIOS DE COLOMBIA

1. ALGORITMO NAÏVE (O(n^4)):
   Tiempo: 0.6865 segundos
   Centro: (-75.863526264+7.1442084475000005j)
   Radio: 1.700077
   Puntos frontera: 2

2. ALGORITMO DE WELZL (O(n) esperado):
   Tiempo: 0.0002 segundos
   Centro: (-75.863526264+7.1442084475000005j)
   Radio: 1.700077
   Puntos frontera: 2

ACELERACIÓN: 3046.77x más rápido con Welzl
